# S07 – Spherical particles (source shape labels, publication-based shape rules)

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S07_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S06_Documented_transcription_error_in_a/S06.xlsx"
OUTPUT = "../S07.xlsx"
COMMENTS = "../S07_comments.xlsx"
AUDIT = "../S07_audit.xlsx"
STEP = "S07"
GUL, LABOUTA, NANOPHAROS = "Gul. Et al..", "Labouta Et. Al. ", "Nanopharos"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

4038 records before the step


### 1. Gul et al.: publications describing round particles, or spheres with other shapes, kept as spherical with a comment

In [2]:
df["Shape_Comments"] = None
# the shape label and the comment with the quotation from the publication: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S07_author_shape_decision'])

94 comments applied (S07_author_shape_decision): 94 values set, 0 records removed


### 2. Gul et al.: every other shape label than 'Sphere': removed

In [3]:
n = len(removed)
drop((df._src == GUL) & df["Shape"].astype(str).str.strip().ne("Sphere"), f"{STEP}_gul_shape",
     "source shape label of Gul et al. is not 'Sphere' (label checked against the publication wherever the full text was available)")
print(len(removed) - n, "records removed")

407 records removed


### 3. Labouta et al. has no shape field: the shape of every construct, read from its publication

In [4]:
# the shape of each record from the construct of its publication (label 'Sphere' for spherical particles, as Gul and NanoPharos): the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S07_labouta_shape_rule'])

624 comments applied (S07_labouta_shape_rule): 624 values set, 0 records removed


### 4. Labouta et al.: constructs that are not spherical: removed

In [5]:
n = len(removed)
drop((df._src == LABOUTA) & df["Shape"].astype(str).ne("Sphere"), f"{STEP}_labouta_shape",
     "Labouta et al. has no shape field: the publication describes a non-spherical, mixed or unresolved shape for this construct")
print(len(removed) - n, "records removed")

175 records removed


### 5. NanoPharos: only spherical particles

In [6]:
for i in df.index[df._src == NANOPHAROS]:
    put(i, "Shape", "Sphere", f"{STEP}_nanopharos_shape",
        "NanoPharos database contains only spherical particles (confirmed by the dataset author); Zhang et al. 2012 text gives no morphology")

### The table after the step and its audit

In [7]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

582 records removed and 956 cells changed compared with the table before the step


3456 records after the step | 582 records removed | 1212 cells changed | ../S07_audit.xlsx
